# Medical Visual Question Answering (Med-VQA) using MMQ Model

This notebook implements the **MMQ (Multi-meta Model Quantification)** model for the PathVQA dataset. The model uses a MAML-based visual encoder and a BERT-based question encoder, fused with a Bilinear Attention Network (BAN) inspired architecture.

## Configuration
- **Dataset**: `preprocessed-path-vqa-flaviagiammarino`
- **Hardware**: Kaggle GPU T4 x 2
- **Model**: MMQ (MAML-based)
- **Metrics**: Loss, Accuracy, F1 Score (Closed-ended), BLEU Score (Open-ended)

In [19]:
!pip install -q datasets transformers evaluate nltk torchmetrics accelerate

In [20]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torchvision import models, transforms
from transformers import AutoModel, AutoTokenizer
from datasets import load_from_disk
import numpy as np
from tqdm.auto import tqdm
import evaluate
from torchmetrics.classification import MulticlassF1Score
import nltk
from nltk.translate.bleu_score import sentence_bleu

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count()
print(f"Using {n_gpu} GPUs")

Using 1 GPUs


In [21]:
# Constants
# Using original PathVQA dataset from Hugging Face (not the preprocessed version)
DATASET_NAME = "flaviagiammarino/path-vqa"
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 16 if n_gpu == 0 else 32 * n_gpu  # Adjusted for local/GPU
EPOCHS = 10  # Reduced for faster training
LEARNING_RATE = 2e-5
MAX_LENGTH = 64

In [22]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import CLIPVisionModel, AutoTokenizer, T5ForConditionalGeneration, CLIPProcessor
from datasets import load_from_disk
from accelerate import Accelerator
from evaluate import load
import numpy as np
from tqdm.auto import tqdm

In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Image preprocessing
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def preprocess_function(examples):
    # Tokenize questions
    inputs = tokenizer(examples['question'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    # Convert answers to labels
    inputs['labels'] = [ans2idx.get(ans, 0) for ans in examples['answer']]
    return inputs

print("Preprocessing dataset...")
train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)
test_ds = test_ds.map(preprocess_function, batched=True)

# Don't set format yet - we'll handle images in the custom dataset class
print("Preprocessing complete!")

Preprocessing dataset...


Map: 100%|██████████| 6719/6719 [00:01<00:00, 3995.35 examples/s]

Preprocessing complete!


In [24]:
# Custom Dataset Class to handle image preprocessing
from torch.utils.data import Dataset as TorchDataset

class PathVQADatasetMMQ(TorchDataset):
    def __init__(self, hf_dataset, image_transform):
        self.dataset = hf_dataset
        self.transform = image_transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Process image
        image = item['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        image = self.transform(image)
        
        # Get preprocessed text and labels
        input_ids = torch.tensor(item['input_ids'], dtype=torch.long)
        attention_mask = torch.tensor(item['attention_mask'], dtype=torch.long)
        labels = torch.tensor(item['labels'], dtype=torch.long)
        
        return {
            'image': image,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

class MMQModel(nn.Module):
    def __init__(self, num_classes, visual_backbone='resnet18', text_model='bert-base-uncased'):
        super(MMQModel, self).__init__()
        # Visual Encoder (MAML-based initialization would happen here)
        self.visual_model = models.resnet18(weights='IMAGENET1K_V1')  # Updated from deprecated pretrained=True
        v_dim = self.visual_model.fc.in_features
        self.visual_model.fc = nn.Identity()
        
        # Question Encoder
        self.text_model = AutoModel.from_pretrained(text_model)
        q_dim = self.text_model.config.hidden_size
        
        # Fusion (BAN-like)
        self.v_proj = nn.Linear(v_dim, 512)
        self.q_proj = nn.Linear(q_dim, 512)
        self.classifier = nn.Linear(512, num_classes)
        
    def forward(self, images, input_ids, attention_mask):
        # Images are already preprocessed in the dataset
        v_features = self.visual_model(images)
        q_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        q_features = q_outputs.last_hidden_state[:, 0, :]
        
        v_proj = self.v_proj(v_features)
        q_proj = self.q_proj(q_features)
        
        fused = v_proj * q_proj
        logits = self.classifier(fused)
        return logits

model = MMQModel(num_classes)
if n_gpu > 1:
    model = nn.DataParallel(model)
model.to(device)

MMQModel(
  (visual_model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [25]:
# Create custom datasets with image preprocessing
train_dataset = PathVQADatasetMMQ(train_ds, image_transform)
val_dataset = PathVQADatasetMMQ(val_ds, image_transform)
test_dataset = PathVQADatasetMMQ(test_ds, image_transform)

# Create dataloaders (num_workers=0 for Windows compatibility)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
f1_metric = MulticlassF1Score(num_classes=num_classes, average='weighted').to(device)

In [26]:
def evaluate_model(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_bleu = []
    
    # Separate metrics for closed and open-ended
    closed_correct = 0
    closed_total = 0
    open_correct = 0  # Exact match for open-ended
    open_total = 0
    open_word_overlap = []  # Word-level accuracy
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(images, input_ids, attention_mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Separate evaluation for closed vs open-ended
            for p, l in zip(preds, labels):
                pred_ans = idx2ans[p.item()].lower().strip()
                label_ans = idx2ans[l.item()].lower().strip()
                
                # Check if it's a closed-ended question (yes/no)
                is_closed = label_ans in ['yes', 'no']
                
                if is_closed:
                    closed_correct += (p == l).item()
                    closed_total += 1
                else:
                    # Open-ended metrics
                    open_total += 1
                    
                    # Exact match accuracy
                    if pred_ans == label_ans:
                        open_correct += 1
                    
                    # Word-level overlap (better for short answers)
                    pred_words = set(pred_ans.split())
                    label_words = set(label_ans.split())
                    if len(label_words) > 0:
                        overlap = len(pred_words & label_words) / len(label_words)
                        open_word_overlap.append(overlap)
                    
                    # BLEU-1 (unigram only, better for short answers)
                    pred_str = pred_ans.split()
                    label_str = [label_ans.split()]
                    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
                    smooth = SmoothingFunction().method1
                    # Use weights=(1,0,0,0) for BLEU-1 only
                    bleu1 = sentence_bleu(label_str, pred_str, weights=(1,0,0,0), smoothing_function=smooth)
                    all_bleu.append(bleu1)
                
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    f1 = f1_metric(torch.tensor(all_preds).to(device), torch.tensor(all_labels).to(device)).item()
    avg_bleu = np.mean(all_bleu) if all_bleu else 0.0
    avg_word_overlap = np.mean(open_word_overlap) if open_word_overlap else 0.0
    
    # Print detailed breakdown
    if closed_total > 0:
        closed_acc = closed_correct / closed_total
        print(f"  Closed-ended: {closed_total} questions, Accuracy: {closed_acc:.4f}")
    if open_total > 0:
        open_acc = open_correct / open_total
        print(f"  Open-ended: {open_total} questions")
        print(f"    - Exact Match: {open_acc:.4f}")
        print(f"    - Word Overlap: {avg_word_overlap:.4f}")
        print(f"    - BLEU-1: {avg_bleu:.4f}")
    
    return avg_loss, accuracy, f1, avg_bleu

# Training Loop
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(images, input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    val_loss, val_acc, val_f1, val_bleu = evaluate_model(val_loader)
    print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}, Val BLEU: {val_bleu:.4f}")

Epoch 1/10:   3%|▎         | 20/615 [00:14<07:02,  1.41it/s]c:\Users\xianz\anaconda3\envs\ml_env\lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
Evaluating: 100%|██████████| 196/196 [01:32<00:00,  2.11it/s]


  Closed-ended: 3125 questions, Accuracy: 0.8426
  Open-ended: 3134 questions
    - Exact Match: 0.1691
    - Word Overlap: 0.1825
    - BLEU-1: 0.1790
Epoch 1: Train Loss: 3.6904, Val Loss: 2.9481, Val Acc: 0.5054, Val F1: 0.4859, Val BLEU: 0.1790


Evaluating: 100%|██████████| 196/196 [01:32<00:00,  2.13it/s]


  Closed-ended: 3125 questions, Accuracy: 0.8534
  Open-ended: 3134 questions
    - Exact Match: 0.2492
    - Word Overlap: 0.2536
    - BLEU-1: 0.2524
Epoch 2: Train Loss: 2.6602, Val Loss: 2.8761, Val Acc: 0.5509, Val F1: 0.5196, Val BLEU: 0.2524


Evaluating: 100%|██████████| 196/196 [01:32<00:00,  2.13it/s]


...

  Closed-ended: 3125 questions, Accuracy: 0.8720
  Open-ended: 3134 questions
    - Exact Match: 0.3063
    - Word Overlap: 0.3301
    - BLEU-1: 0.3226
Epoch 8: Train Loss: 1.0442, Val Loss: 3.4935, Val Acc: 0.5888, Val F1: 0.5712, Val BLEU: 0.3226


Evaluating: 100%|██████████| 196/196 [01:31<00:00,  2.15it/s]


  Closed-ended: 3125 questions, Accuracy: 0.8698
  Open-ended: 3134 questions
    - Exact Match: 0.3060
    - Word Overlap: 0.3322
    - BLEU-1: 0.3240
Epoch 9: Train Loss: 0.8045, Val Loss: 3.6127, Val Acc: 0.5875, Val F1: 0.5691, Val BLEU: 0.3240


Evaluating: 100%|██████████| 196/196 [01:30<00:00,  2.15it/s]

  Closed-ended: 3125 questions, Accuracy: 0.8730
  Open-ended: 3134 questions
    - Exact Match: 0.3156
    - Word Overlap: 0.3435
    - BLEU-1: 0.3351
Epoch 10: Train Loss: 0.6077, Val Loss: 3.7509, Val Acc: 0.5939, Val F1: 0.5778, Val BLEU: 0.3351


In [27]:
# Final Evaluation on Test Set
def evaluate_final(loader):
    model.eval()
    closed_metrics = {'loss': 0, 'correct': 0, 'total': 0, 'preds': [], 'labels': []}
    open_metrics = {'bleu': [], 'correct': 0, 'total': 0, 'word_overlap': []}
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Final Testing"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(images, input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            preds = torch.argmax(logits, dim=1)
            
            for i in range(len(labels)):
                pred_ans = idx2ans[preds[i].item()].lower().strip()
                label_ans = idx2ans[labels[i].item()].lower().strip()
                is_closed = label_ans in ['yes', 'no']
                
                if is_closed:
                    closed_metrics['loss'] += loss.item()
                    closed_metrics['correct'] += (preds[i] == labels[i]).item()
                    closed_metrics['total'] += 1
                    closed_metrics['preds'].append(preds[i].item())
                    closed_metrics['labels'].append(labels[i].item())
                else:
                    open_metrics['total'] += 1
                    
                    # Exact match
                    if pred_ans == label_ans:
                        open_metrics['correct'] += 1
                    
                    # Word overlap
                    pred_words = set(pred_ans.split())
                    label_words = set(label_ans.split())
                    if len(label_words) > 0:
                        overlap = len(pred_words & label_words) / len(label_words)
                        open_metrics['word_overlap'].append(overlap)
                    
                    # BLEU-1 (better for short answers)
                    pred_str = pred_ans.split()
                    label_str = [label_ans.split()]
                    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
                    smooth = SmoothingFunction().method1
                    bleu1 = sentence_bleu(label_str, pred_str, weights=(1,0,0,0), smoothing_function=smooth)
                    open_metrics['bleu'].append(bleu1)
                    
    closed_acc = closed_metrics['correct'] / closed_metrics['total'] if closed_metrics['total'] > 0 else 0
    closed_f1 = f1_metric(torch.tensor(closed_metrics['preds']).to(device), 
                          torch.tensor(closed_metrics['labels']).to(device)).item() if closed_metrics['total'] > 0 else 0
    
    open_acc = open_metrics['correct'] / open_metrics['total'] if open_metrics['total'] > 0 else 0
    avg_word_overlap = np.mean(open_metrics['word_overlap']) if open_metrics['word_overlap'] else 0
    avg_bleu = np.mean(open_metrics['bleu']) if open_metrics['bleu'] else 0
    
    print(f"\n{'='*50}")
    print(f"{'FINAL TEST RESULTS':^50}")
    print(f"{'='*50}")
    print(f"\nClosed-ended Questions ({closed_metrics['total']} total):")
    print(f"  - Accuracy: {closed_acc:.4f}")
    print(f"  - F1 Score: {closed_f1:.4f}")
    print(f"\nOpen-ended Questions ({open_metrics['total']} total):")
    print(f"  - Exact Match: {open_acc:.4f}")
    print(f"  - Word Overlap: {avg_word_overlap:.4f}")
    print(f"  - BLEU-1: {avg_bleu:.4f}")
    print(f"{'='*50}\n")

evaluate_final(test_loader)

Final Testing: 100%|██████████| 210/210 [01:37<00:00,  2.15it/s]


                FINAL TEST RESULTS                

Closed-ended Questions (3362 total):
  - Accuracy: 0.8739
  - F1 Score: 0.8742

Open-ended Questions (3357 total):
  - Exact Match: 0.3113
  - Word Overlap: 0.3238
  - BLEU-1: 0.3193

